In [15]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
from tqdm import tqdm

def setup_llm_router(model_id="Qwen/Qwen2.5-3B-Instruct"):
    """
    LLM과 토크나이저를 VRAM에 안전하게 로드합니다.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )
    return model, tokenizer

def classify_with_llm(file_path, model, tokenizer):
    if not os.path.exists(file_path):
        print(f"파일을 찾을 수 없습니다: {file_path}")
        return
        
    df = pd.read_csv(file_path).head(100)
    results = []
    
    print(f"[{file_path}] LLM 라우팅 추론 시작...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        q_text = str(row['question']) if pd.notna(row['question']) else ""
        
        # 🚨 [수정 포인트 1] a, b, c, d 알파벳을 제거하여 객관식 문제처럼 보이지 않게 위장
        c_text = f"{row.get('a','')} / {row.get('b','')} / {row.get('c','')} / {row.get('d','')}"
        
        # 🚨 [수정 포인트 2] 시스템 프롬프트를 줄이고 3개의 가짜 모범답안(Few-Shot) 주입
        messages = [
            {"role": "system", "content": "당신은 VQA 라우터입니다. 오직 'YOLO' 또는 'VLM'만 출력합니다. 문제를 절대 직접 풀지 마세요."},
            
            # [모범 답안 1] 개수 세기
            {"role": "user", "content": "질문: 사진에 보이는 플라스틱 병은 몇 개입니까?\n보기: 1개 / 2개 / 3개 / 4개\n분류결과:"},
            {"role": "assistant", "content": "YOLO"},
            
            # [모범 답안 2] 함정 방어 (무게/OCR)
            {"role": "user", "content": "질문: 상자에 적힌 무게는 얼마인가요?\n보기: 10kg / 5kg / 13개 / 30개\n분류결과:"},
            {"role": "assistant", "content": "VLM"},
            
            # [모범 답안 3] 종류/재질 묻기
            {"role": "user", "content": "질문: 음료 캔의 종류는 무엇인가요?\n보기: 콜라 / 사이다 / 환타 / 맥주\n분류결과:"},
            {"role": "assistant", "content": "VLM"},
            
            # [실제 풀게 될 문제]
            {"role": "user", "content": f"질문: {q_text}\n보기: {c_text}\n분류결과:"}
        ]
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=3,  # 🚨 [수정 포인트 3] 토큰 수를 3으로 제한하여 딴소리 원천 차단
                temperature=0.01,
                do_sample=False
            )
        
        generated_ids = outputs[0][len(inputs.input_ids[0]):]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        
        if "YOLO" in response.upper():
            final_class = "YOLO"
        else:
            final_class = "VLM" 
            
        results.append(final_class)
        
    df['llm_class'] = results
    new_file_path = file_path.replace('.csv', '_class.csv')
    df.to_csv(new_file_path, index=False, encoding='utf-8-sig')
    print(f"저장 완료: {new_file_path} (총 {len(df)}행)\n")

In [16]:
# 메인 실행 로직
if __name__ == "__main__":
    print("한 단계 업그레이드된 3B LLM 라우터 모델을 로드하는 중입니다...")
    
    # 🚨 [수정 포인트 4] 체급을 0.5B에서 3B로 상향 (5060Ti에서 VRAM 약 6GB 내외 사용)
    model, tokenizer = setup_llm_router("Qwen/Qwen2.5-3B-Instruct")
    
    target_files = ['train.csv', 'dev.csv', 'test.csv']
    
    for file in target_files:
        classify_with_llm(file, model, tokenizer)
        
    print("모든 파일의 지능형 LLM 라우팅이 완료되었습니다!")

한 단계 업그레이드된 3B LLM 라우터 모델을 로드하는 중입니다...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


[train.csv] LLM 라우팅 추론 시작...


100% 100/100 [04:19<00:00,  2.59s/it]


저장 완료: train_class.csv (총 100행)

[dev.csv] LLM 라우팅 추론 시작...


100% 100/100 [04:19<00:00,  2.60s/it]


저장 완료: dev_class.csv (총 100행)

[test.csv] LLM 라우팅 추론 시작...


  0% 0/100 [00:00<?, ?it/s]


KeyboardInterrupt: 